In [ ]:
!pip -q install groq sentence-transformers xgboost numpy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.6 MB/s eta 0:00:00


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"


In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb
from sentence_transformers import SentenceTransformer
from groq import Groq

# -------------------------
# GROQ API KEY
# -------------------------
# Recommended: set GROQ_API_KEY in Colab "Secrets" panel (left sidebar).
# Or (not recommended for shared notebooks):
# os.environ["GROQ_API_KEY"] = "YOUR_GROQ_KEY"
# Create Groq client (reads GROQ_API_KEY from environment)
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])  # reads GROQ_API_KEY from env

# -------------------------
# GROQ MODEL (must exist on your account)
# -------------------------
# Groq model id (must exist in your account)
# If you get model_not_found, run Cell 3 and pick an id printed there.
GROQ_MODEL_ID = "llama-3.1-8b-instant"

# -------------------------
# SBERT (must match training)
# -------------------------
SBERT_NAME = "all-MiniLM-L6-v2"   #  SBERT
NORMALIZE_EMB = False            # set True if you used normalize_embeddings=True during training

# -------------------------
#  critic models (upload to /content)
# -------------------------
MODEL_PATHS = {
    "STANCE": "/content/xgb_output_0.json",
    "ACTION": "/content/xgb_output_1.json",
    "PERSONALNESS": "/content/xgb_output_2.json",
    "POLITENESS": "/content/xgb_output_3.json",
}

LABELS = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]

# -------------------------
# Threshold logic
# -------------------------
# We compute a score in [0,1]. If score >= threshold -> accept.
OVERALL_SCORE_THRESHOLD = 0.75
PER_LABEL_THRESHOLD = 0.75

# -------------------------
# Search settings
# -------------------------
N_CANDIDATES = 12
MAX_ROUNDS = 3
MAX_TOKENS = 150
TEMPS = np.linspace(0.4, 1.0, N_CANDIDATES)

# Optional weights (increase importance if needed)
WEIGHTS = {"STANCE": 1.0, "ACTION": 1.0, "PERSONALNESS": 1.0, "POLITENESS": 1.0}


In [ ]:
# ================================
# Cell 3: (Optional) List accessible Groq models
# Run this if you get "model_not_found"
# ================================

models = groq_client.models.list()
for m in models.data:
    print(m.id)


allam-2-7b
openai/gpt-oss-20b
groq/compound-mini
moonshotai/kimi-k2-instruct
qwen/qwen3-32b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
meta-llama/llama-guard-4-12b
openai/gpt-oss-safeguard-20b
moonshotai/kimi-k2-instruct-0905
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
groq/compound
whisper-large-v3
llama-3.1-8b-instant
canopylabs/orpheus-arabic-saudi
meta-llama/llama-4-maverick-17b-128e-instruct
meta-llama/llama-prompt-guard-2-22m
llama-3.3-70b-versatile


In [ ]:
# SBERT
sbert = SentenceTransformer(SBERT_NAME)
print("SBERT loaded:", SBERT_NAME)

# Critics
critics = {}
for name, path in MODEL_PATHS.items():
    model = xgb.XGBClassifier()
    model.load_model(path)
    critics[name] = model

print("Loaded critics:", list(critics.keys()))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT loaded: all-MiniLM-L6-v2
Loaded critics: ['STANCE', 'ACTION', 'PERSONALNESS', 'POLITENESS']


In [ ]:

# ================================
# Cell 5: LLM generation wrapper (Groq -> LLaMA)
# This is the ONLY place the LLM is called.
# ================================
def llm_generate_reply(target_tweet: str, context: str, temperature: float = 0.8) -> str:
    prompt = f"""{context}

Target tweet:
{target_tweet}

Write ONE reply only. Keep it 1–3 sentences.
"""

    chat = groq_client.chat.completions.create(
        model=GROQ_MODEL_ID,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=MAX_TOKENS,
    )
    return chat.choices[0].message.content.strip()


In [ ]:
# ================================
# Cell 6: Embedding + critic scoring functions
# - embed_combo: SBERT embeds (tweet + reply)
# - critics_have_proba: check if models support predict_proba
# - p_gold_scores: probability of GOLD class per label (requires softprob/logistic)
# - match_score: fallback if proba not available
# ================================

def embed_combo(target_tweet: str, reply: str) -> np.ndarray:
    combo = (str(target_tweet) + " " + str(reply)).strip()
    emb = sbert.encode(combo, normalize_embeddings=NORMALIZE_EMB)
    return np.asarray(emb, dtype=np.float32).reshape(1, -1)

def critics_have_proba() -> bool:
    return all(hasattr(m, "predict_proba") for m in critics.values())

def predict_discrete(target_tweet: str, reply: str) -> dict:
    X = embed_combo(target_tweet, reply)
    return {k: int(m.predict(X)[0]) for k, m in critics.items()}

def p_gold_scores(target_tweet: str, reply: str, gold: dict) -> dict:
    """
    p_gold[label] = probability assigned to the GOLD class for that label.
    Works properly only if the model was trained with:
      - multiclass: objective="multi:softprob"
      - binary: objective="binary:logistic"
    If your objective was "multi:softmax", probabilities won't be available.
    """
    X = embed_combo(target_tweet, reply)
    out = {}
    for label, model in critics.items():
        proba = model.predict_proba(X)[0]
        g = int(gold[label])
        if g < 0 or g >= len(proba):
            raise ValueError(f"Gold id {g} out of range for {label}. num_classes={len(proba)}")
        out[label] = float(proba[g])
    return out

def weighted_avg(values: dict, weights: dict) -> float:
    num, den = 0.0, 0.0
    for k, v in values.items():
        w = float(weights.get(k, 1.0))
        num += w * float(v)
        den += w
    return num / den if den > 0 else 0.0

def match_score(pred: dict, gold: dict, weights: dict) -> float:
    """
    Fallback score in [0,1] if probabilities are not available.
    """
    num, den = 0.0, 0.0
    for k in LABELS:
        w = float(weights.get(k, 1.0))
        den += w
        num += w * (1.0 if int(pred[k]) == int(gold[k]) else 0.0)
    return num / den if den > 0 else 0.0


In [ ]:
# ================================
# Cell 7: Prompt logic (RLHF-like prompt steering)
# - build_base_context: base prompt with constraints
# - tighten_context: adds constraints for weak labels (low probs) or mismatches
# ================================

def build_base_context(gold: dict) -> str:
    return (
        "Reply naturally to the tweet.\n"
        "Constraints:\n"
        "- One reply only (no multiple options).\n"
        "- 1–3 sentences.\n"
        "- Do not mention labels, critics, loss, or probabilities.\n"
        f"- Target pragmatic classes (internal): {gold}\n"
    )

def tighten_context(context: str, gold: dict, signal: dict, per_label_threshold: float, proba_mode: bool) -> str:
    """
    If proba_mode=True: signal is p_gold dict -> tighten labels with p < threshold.
    If proba_mode=False: signal is pred dict -> tighten mismatched labels.
    """
    if proba_mode:
        weak = [k for k, v in signal.items() if float(v) < per_label_threshold]
    else:
        weak = [k for k in LABELS if int(signal[k]) != int(gold[k])]

    if not weak:
        return context

    extra = "\nSTRICT rewrite constraints (must follow):\n"
    for k in weak:
        extra += f"- Ensure {k} matches gold class id {gold[k]}.\n"
    extra += "- Rewrite the reply to satisfy these constraints while staying coherent.\n"
    extra += "- Keep 1–3 sentences. Do not mention constraints.\n"
    return context + extra


In [ ]:
# ================================
# Cell 8: Main generation loop
# - Generate N candidates each round
# - Score each candidate using critics
# - Keep the best
# - If score >= threshold -> accept
# - Else -> tighten prompt and repeat
# ================================


def generate_reply_with_threshold(
    target_tweet: str,
    gold: dict,
    overall_threshold: float = OVERALL_SCORE_THRESHOLD,
    per_label_threshold: float = PER_LABEL_THRESHOLD,
    n_candidates: int = N_CANDIDATES,
    max_rounds: int = MAX_ROUNDS,
):
    proba_mode = critics_have_proba()
    context = build_base_context(gold)

    best_reply, best_score, best_signal = None, -1.0, None

    for r in range(max_rounds):
        candidates = []

        for temp in np.linspace(0.4, 1.0, n_candidates):
            reply = llm_generate_reply(target_tweet, context, temperature=float(temp))

            if proba_mode:
                signal = p_gold_scores(target_tweet, reply, gold)     # dict of gold probs
                score = weighted_avg(signal, WEIGHTS)                 # 0..1
            else:
                signal = predict_discrete(target_tweet, reply)         # dict of class ids
                score = match_score(signal, gold, WEIGHTS)             # 0..1

            candidates.append((score, reply, signal))

        candidates.sort(key=lambda x: x[0], reverse=True)
        score, reply, signal = candidates[0]

        if score > best_score:
            best_score, best_reply, best_signal = score, reply, signal

        print(f"\nRound {r+1}/{max_rounds}")
        print(f"Best score: {score:.3f} (threshold={overall_threshold})")
        print("Signal:", signal)
        print("Reply:", reply)

        # ✅ If score is good enough, keep it
        if score >= overall_threshold:
            return reply, score, signal

        # ❌ Otherwise tighten prompt and retry
        context = tighten_context(context, gold, signal, per_label_threshold, proba_mode)

    return best_reply, best_score, best_signal


In [ ]:
# ================================
# Cell 9: Load LabelEncoders (must match training)
# This is required when your dataset labels are strings like "SUPPORT"
# ================================
import joblib

encoders = joblib.load("/content/label_encoders.pkl")


In [ ]:
# ================================
# Cell 10: Load dataset and pick one sample
# - Reads an Excel file
# - Extracts tweet text + gold labels (encoded)
# ================================

CSV_PATH = "/content/Annotated_Data.xlsx"  # upload your csv to /content
df = pd.read_excel(CSV_PATH)

# Change these column names to match your CSV
TWEET_COL = "target_tweet"
GOLD_COLS = {"STANCE":"STANCE", "ACTION":"ACTION", "PERSONALNESS":"PERSONALNESS", "POLITENESS":"POLITENESS"}

i = 0  # choose which row
row = df.iloc[i]

target_tweet = str(row[TWEET_COL])

gold = {
    label: int(encoders[label].transform([row[col]])[0])
    for label, col in GOLD_COLS.items()
}

print("Gold (encoded):", gold)



Gold (encoded): {'STANCE': 2, 'ACTION': 3, 'PERSONALNESS': 0, 'POLITENESS': 1}


In [ ]:
# ================================
# Cell 11: Run generation + critics + threshold loop
# ================================
target_tweet = "I can't believe the prices keep rising. This is getting ridiculous."
gold = {"STANCE": 2, "ACTION": 1, "PERSONALNESS": 0, "POLITENESS": 2}

print("Tweet:", target_tweet)
print("Gold:", gold)


Tweet: I can't believe the prices keep rising. This is getting ridiculous.
Gold: {'STANCE': 2, 'ACTION': 1, 'PERSONALNESS': 0, 'POLITENESS': 2}


In [ ]:
# ================================
# Cell 11: Run generation + critics + threshold loop
# ================================
best_reply, best_score, best_signal = generate_reply_with_threshold(
    target_tweet=target_tweet,
    gold=gold,
    overall_threshold=0.75,
    per_label_threshold=0.75,
    n_candidates=12,
    max_rounds=3,
)

print("\n=== FINAL OUTPUT ===")
print("Best score:", best_score)
print("Best signal:", best_signal)
print("Best reply:", best_reply)



Round 1/3
Best score: 0.206 (threshold=0.75)
Signal: {'STANCE': 0.09643851965665817, 'ACTION': 0.0007958923233672976, 'PERSONALNESS': 0.7229233384132385, 'POLITENESS': 0.003461903426796198}
Reply: I feel you, it's getting tough to keep up with the cost of living. Maybe we can look into some budgeting apps to help us save some money?

Round 2/3
Best score: 0.174 (threshold=0.75)
Signal: {'STANCE': 0.02850530669093132, 'ACTION': 0.002044174587354064, 'PERSONALNESS': 0.6546703577041626, 'POLITENESS': 0.011096187867224216}
Reply: It's really tough when budgets get stretched so thin, but I'm sure we'll find a way to manage. Maybe it's time to reevaluate our spending habits and see where we can cut back?

Round 3/3
Best score: 0.164 (threshold=0.75)
Signal: {'STANCE': 0.03287405148148537, 'ACTION': 0.0025062228087335825, 'PERSONALNESS': 0.6007660627365112, 'POLITENESS': 0.02039177529513836}
Reply: We should start a petition to bring attention to this issue and see if we can get some relief.